In [1]:
import json

import numpy as np
import tensorflow as tf
from keras.layers import TextVectorization
from tensorflow import Tensor
from tqdm import tqdm


# =============================================================================
# PATHS
# =============================================================================

CAPTIONS_FILE = "../data/captions/captions_split_clean.json"
OUTPUT_DIR = "../data/records"


# =============================================================================
# LOAD DATA
# =============================================================================

def load_captions(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# =============================================================================
# PREPROCESSING
# =============================================================================

def compute_max_len(captions: dict[str, list[str]], percentile: int = 97) -> int:
    lengths = [
        len(c.split())
        for caps in captions.values()
        for c in caps
    ]
    return int(np.percentile(lengths, percentile))


def truncate_captions(
    captions: dict[str, list[str]],
    max_len: int,
) -> tuple[dict[str, list[str]], int]:

    out = {}
    count = 0

    for img, caps in captions.items():
        processed = []

        for c in caps:
            words = c.split()

            if len(words) > max_len:
                processed.append(" ".join(words[:max_len]))
                count += 1
            else:
                processed.append(c)

        out[img] = processed

    return out, count


def add_start_end_tokens(captions: dict[str, list[str]]) -> dict[str, list[str]]:

    return {
        img: [f"[START] {c} [END]" for c in caps]
        for img, caps in captions.items()
    }


# =============================================================================
# VECTORIZE
# =============================================================================

def create_vectorizer(
    captions: dict[str, list[str]],
    output_sequence_length: int,
) -> TextVectorization:

    vectorizer = TextVectorization(
        output_mode="int",
        split="whitespace",
        standardize=None, # type: ignore
        output_sequence_length=output_sequence_length,
        name="text_vectorization",
    )

    flat = [c for caps in captions.values() for c in caps]
    vectorizer.adapt(flat)

    return vectorizer


def vectorize_captions(
    vectorizer: TextVectorization,
    captions: dict[str, list[str]],
) -> Tensor:

    flat = [c for caps in captions.values() for c in caps]
    return vectorizer(flat)


def split_teacher_forcing(captions: Tensor) -> tuple[Tensor, Tensor]:
    return captions[:, :-1], captions[:, 1:] # type: ignore


# =============================================================================
# DATASET
# =============================================================================

def build_dataset(
    captions: dict[str, list[str]],
    input_tensor: Tensor,
    target_tensor: Tensor,
):

    dataset = []
    idx = 0

    for img, caps in captions.items():
        for _ in caps:
            dataset.append({
                "image": img,
                "input": input_tensor[idx], # type: ignore
                "target": target_tensor[idx], # type: ignore
            })
            idx += 1

    return dataset


# =============================================================================
# TFRECORD
# =============================================================================

def create_tfrecord(dataset, output_file: str) -> None:

    with tf.io.TFRecordWriter(output_file) as writer:

        for sample in tqdm(dataset, desc="Writing TFRecord"):

            example = tf.train.Example(
                features=tf.train.Features(
                    feature={
                        "image": tf.train.Feature(
                            bytes_list=tf.train.BytesList(
                                value=[sample["image"].encode()]
                            )
                        ),
                        "input": tf.train.Feature(
                            int64_list=tf.train.Int64List(
                                value=sample["input"].numpy().astype(np.int64).tolist()
                            )
                        ),
                        "target": tf.train.Feature(
                            int64_list=tf.train.Int64List(
                                value=sample["target"].numpy().astype(np.int64).tolist()
                            )
                        ),
                    }
                )
            )

            writer.write(example.SerializeToString()) # type: ignore


# =============================================================================
# PIPELINE
# =============================================================================

def process_split(
    split_name: str,
    captions: dict,
    vectorizer: TextVectorization | None,
    max_len: int,
):

    print(f"\n=== Processing {split_name} ===")

    captions = captions[split_name]

    captions, truncated = truncate_captions(captions, max_len)
    print(f"{split_name}: truncated {truncated} captions")

    captions = add_start_end_tokens(captions)

    if vectorizer is None:
        vectorizer = create_vectorizer(captions, max_len + 2) # [START and [END] tokens
        print(f"Vocabulary size: {len(vectorizer.get_vocabulary())}")

    tensor = vectorize_captions(vectorizer, captions)

    inp, tgt = split_teacher_forcing(tensor)

    dataset = build_dataset(captions, inp, tgt)

    create_tfrecord(dataset, f"{OUTPUT_DIR}/{split_name}.tfrecord")

    print(f"Saved: {OUTPUT_DIR}/{split_name}.tfrecord")

    return dataset, vectorizer


# =============================================================================
# MAIN
# =============================================================================

captions = load_captions(CAPTIONS_FILE)

train_captions = captions["train"]

max_len = compute_max_len(train_captions, percentile=97)
print(f"Max length: {max_len}")

train_dataset, vectorizer = process_split(
    "train",
    captions,
    None,
    max_len,
)

val_dataset, _ = process_split(
    "val",
    captions,
    vectorizer,
    max_len,
)

test_dataset, _ = process_split(
    "test",
    captions,
    vectorizer,
    max_len,
)

print("\nTFRecords created successfully.")

2026-06-29 17:46:01.252492: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Max length: 19

=== Processing train ===
train: truncated 3569 captions
Vocabulary size: 18028


Writing TFRecord: 100%|██████████| 127130/127130 [00:03<00:00, 35842.74it/s]


Saved: ../data/records/train.tfrecord

=== Processing val ===
val: truncated 434 captions


Writing TFRecord: 100%|██████████| 15890/15890 [00:00<00:00, 36660.11it/s]


Saved: ../data/records/val.tfrecord

=== Processing test ===
test: truncated 404 captions


Writing TFRecord: 100%|██████████| 15895/15895 [00:00<00:00, 37092.87it/s]

Saved: ../data/records/test.tfrecord

TFRecords created successfully.
